In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electronics_retailer_clg.silver;

In [0]:
from pyspark.sql.functions import col, trim, when

# ================================
# 1. READ BRONZE TABLE
# ================================

df = spark.table("electronics_retailer_clg.bronze.customers")


# ================================
# 2. CLEAN COLUMN NAMES
# ================================

df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])


# ================================
# 3. TRIM SPACES
# ================================

for c in df.columns:
    df = df.withColumn(c, trim(col(c)))


# ================================
# 4. REMOVE INVALID PRIMARY KEYS
# ================================

df = df.filter(col("customerkey").isNotNull())


# ================================
# 5. FIX DATA TYPES
# ================================

df = df.withColumn("customerkey", col("customerkey").cast("int"))


# ================================
# 6. HANDLE NULL VALUES
# ================================

df = df.fillna({
    "gender": "unknown",
    "continent": "unknown"
})


# ================================
# 7. STANDARDIZE GENDER
# ================================

df = df.withColumn(
    "gender",
    when(col("gender").isin("Male", "Female"), col("gender"))
    .otherwise("Unknown")
)


# ================================
# 8. REMOVE DUPLICATES
# ================================

df = df.dropDuplicates(["customerkey"])


# ================================
# 9. KEEP ONLY REQUIRED COLUMNS
# ================================

df = df.select(
    "customerkey",
    "gender",
    "continent"
)


# ================================
# 10. FINAL CHECK
# ================================

display(df)
df.printSchema()


# ================================
# 11. WRITE TO SILVER
# ================================

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("electronics_retailer_clg.silver.customers")

print("✅ Customers cleaned & optimized successfully")